# 06 - Interpretability With End-To-End EfficientNetB3

Train an end-to-end `ArcFaceModel` with an unfrozen EfficientNetB3 backbone and visualize full-model Layer-wise Relevance Propagation (LRP) heatmaps on validation images.

## Notes

- This notebook no longer uses cached embeddings. The full EfficientNetB3 backbone and projection head are trained jointly.
- Attribution is computed on the complete image-to-class-score model, so relevance is propagated through the backbone as well as the ArcFace projection head.
- The LRP section uses `captum.attr.LRP`. If Captum is not installed in your environment, run `%pip install captum` in a notebook cell first.
- The attribution target is the cosine class score induced by the trained ArcFace weights, not the margin-modified training logit.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm.data
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from dotenv import load_dotenv
from torch.utils.data import DataLoader

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.training as training
import src.wandb_utils as wandb_utils
from src.interpretability import (
    ArcFaceClassifierForLRP,
    lrp_attribution_to_heatmap,
)
from src.models import ArcFaceModel
from src.utils import get_device, set_seed

load_dotenv(project_root / '.env')

RANDOM_SEED = 42
set_seed(RANDOM_SEED)
device = get_device()
device


## Config

In [ ]:
config = {
    'data_dir': Path('../data'),
    'checkpoint_dir': Path('../checkpoints/e6_interpretability'),
    'backbone_name': 'efficientnet_b3',
    'input_size': 300,
    'embedding_dim': 256,
    'hidden_dim': 512,
    'dropout': 0.3,
    'freeze_backbone': False,
    'arcface_margin': 0.5,
    'arcface_scale': 64.0,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'head_learning_rate': 1e-4,
    'backbone_learning_rate': 1e-5,
    'weight_decay': 1e-4,
    'num_epochs': 30,
    'patience': 5,
    'val_split': 0.2,
    'num_workers': 2,
    'seed': RANDOM_SEED,
    'use_wandb': True,
}

config['checkpoint_dir'].mkdir(parents=True, exist_ok=True)
config


## Load Data Split

In [ ]:
train_df = data.load_train_df(config['data_dir'])
train_df, label_encoder = data.encode_labels(train_df)
num_classes = len(label_encoder.classes_)

train_data, val_data = data.train_val_split(
    train_df,
    val_split=config['val_split'],
    seed=config['seed'],
    stratify_col='ground_truth',
)

print(f'Train samples: {len(train_data)}')
print(f'Val samples:   {len(val_data)}')
print(f'Num classes:   {num_classes}')

## Build End-To-End Model

In [ ]:
model = ArcFaceModel(
    backbone_name=config['backbone_name'],
    num_classes=num_classes,
    embedding_dim=config['embedding_dim'],
    hidden_dim=config['hidden_dim'],
    margin=config['arcface_margin'],
    scale=config['arcface_scale'],
    dropout=config['dropout'],
    pretrained=True,
    freeze_backbone=config['freeze_backbone'],
).to(device)

data_config = timm.data.resolve_model_data_config(model.backbone)
mean = data_config['mean']
std = data_config['std']

train_loader, val_loader = data.create_dataloaders(
    train_df=train_data,
    val_df=val_data,
    img_dir=config['data_dir'] / 'train' / 'train',
    input_size=config['input_size'],
    batch_size=config['batch_size'],
    num_workers=config['num_workers'],
    mean=mean,
    std=std,
    weighted_sampling=False,
    label_col='label_encoded',
    augment=False,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')


## Train ArcFaceModel

## Optional W&B Logging


In [ ]:
wandb_run = None
if config.get('use_wandb', False):
    run_name = '06-EfficientNetB3-ArcFace-EndToEnd-LRP'
    backbone_param_count = sum(p.numel() for p in model.backbone.parameters())
    param_count = sum(p.numel() for p in model.parameters())
    wandb_run = wandb_utils.init_wandb(
        run_config=config,
        run_name=run_name,
        param_count=param_count,
        param_count_backbone=backbone_param_count,
    )
    wandb_run.config.update({'num_classes': num_classes}, allow_val_change=True)
wandb_run


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = training.build_optimizer(model, config)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

results = training.fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    label_encoder=label_encoder,
    wandb_run=wandb_run,
    checkpoint_name='efficientnet_b3_arcface_end_to_end.pth',
)

if wandb_run is not None:
    wandb_run.finish()

results['best_map'], results['best_epoch'], results['epochs_ran']


## Training Curves

In [ ]:
history = pd.DataFrame(results['history'])
history.index = np.arange(1, len(history) + 1)
history.index.name = 'epoch'
history.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history.index, history['train_loss'], label='train')
axes[0].plot(history.index, history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.index, history['train_acc'], label='train')
axes[1].plot(history.index, history['val_acc'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history.index, history['val_map'], label='val mAP', color='tab:green')
axes[2].set_title('Validation mAP')
axes[2].legend()

for ax in axes:
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Build Full-Model Classifier For LRP

In [ ]:
classifier = ArcFaceClassifierForLRP(model).to(device)
classifier.eval()

## Captum Setup

In [ ]:
try:
    from captum.attr import LRP
    from captum.attr._utils.lrp_rules import EpsilonRule
except ImportError as exc:
    raise ImportError(
        'Captum is required for full-model LRP. Install it with `%pip install captum` and rerun this notebook.'
    ) from exc

def attach_epsilon_rules(module):
    supported = (
        nn.Conv2d,
        nn.Linear,
        nn.BatchNorm2d,
        nn.BatchNorm1d,
        nn.AvgPool2d,
        nn.AdaptiveAvgPool2d,
    )
    for child in module.modules():
        if isinstance(child, supported):
            child.rule = EpsilonRule(epsilon=1e-6)

attach_epsilon_rules(classifier)
lrp = LRP(classifier)

## Attribution Helpers

In [ ]:
image_dir = config['data_dir'] / 'train' / 'train'
transform = data.build_transforms_baseline(
    input_size=config['input_size'],
    train=False,
    mean=mean,
    std=std,
    augment=False,
)

def load_image_tensor(filename):
    image = Image.open(image_dir / filename).convert('RGB')
    tensor = transform(image)
    return image, tensor

def predict_row(row):
    _, tensor = load_image_tensor(row['filename'])
    logits = classifier(tensor.unsqueeze(0).to(device))
    pred_idx = int(logits.argmax(dim=1).item())
    pred_score = float(logits.max(dim=1).values.item())
    return pred_idx, pred_score

def compute_lrp_result(filename, target_idx=None):
    image, tensor = load_image_tensor(filename)
    input_batch = tensor.unsqueeze(0).to(device)
    input_batch.requires_grad_(True)

    logits = classifier(input_batch)
    if target_idx is None:
        target_idx = int(logits.argmax(dim=1).item())

    attribution = lrp.attribute(input_batch, target=target_idx)
    heatmap = lrp_attribution_to_heatmap(attribution)[0].detach().cpu().numpy()
    return {
        'image': image,
        'heatmap': heatmap,
        'pred_idx': target_idx,
        'pred_label': label_encoder.inverse_transform([target_idx])[0],
        'pred_score': float(logits[0, target_idx].item()),
    }

def show_lrp_result(result, ax_image, ax_heatmap):
    image_np = np.asarray(result['image'])
    heatmap = result['heatmap']

    ax_image.imshow(image_np)
    ax_image.set_title(
        f"Pred: {result['pred_label']}\nscore={result['pred_score']:.3f}"
    )
    ax_image.axis('off')

    ax_heatmap.imshow(image_np)
    ax_heatmap.imshow(heatmap, cmap='inferno', alpha=0.45)
    ax_heatmap.set_title('Full-model LRP')
    ax_heatmap.axis('off')


## Score Validation Images

In [ ]:
val_examples = val_data.copy().reset_index(drop=True)
preds = val_examples.apply(predict_row, axis=1, result_type='expand')
preds.columns = ['pred_idx', 'pred_score']
val_examples = pd.concat([val_examples, preds], axis=1)
val_examples['pred_label'] = label_encoder.inverse_transform(val_examples['pred_idx'])
val_examples['is_correct'] = val_examples['pred_idx'] == val_examples['label_encoded']

print(f'Validation top-1 accuracy on scored samples: {val_examples["is_correct"].mean():.3f}')
val_examples[['filename', 'ground_truth', 'pred_label', 'pred_score', 'is_correct']].head()

## LRP On Correct Validation Predictions

In [ ]:
selected_examples = (
    val_examples[val_examples['is_correct']]
    .sort_values(['pred_score', 'filename'], ascending=[False, True])
    .head(6)
    .reset_index(drop=True)
)

selected_examples[['filename', 'ground_truth', 'pred_label', 'pred_score']]

In [ ]:
fig, axes = plt.subplots(len(selected_examples), 2, figsize=(10, 4 * len(selected_examples)))
if len(selected_examples) == 1:
    axes = np.array([axes])

for row_idx, row in selected_examples.iterrows():
    result = compute_lrp_result(row['filename'], target_idx=int(row['pred_idx']))
    show_lrp_result(result, axes[row_idx, 0], axes[row_idx, 1])

plt.tight_layout()
plt.show()

## Optional: LRP On Misclassifications

In [ ]:
mistakes = (
    val_examples[~val_examples['is_correct']]
    .sort_values(['pred_score', 'filename'], ascending=[False, True])
    .head(4)
    .reset_index(drop=True)
)
mistakes[['filename', 'ground_truth', 'pred_label', 'pred_score']]

In [ ]:
if len(mistakes) > 0:
    fig, axes = plt.subplots(len(mistakes), 2, figsize=(10, 4 * len(mistakes)))
    if len(mistakes) == 1:
        axes = np.array([axes])

    for row_idx, row in mistakes.iterrows():
        result = compute_lrp_result(row['filename'], target_idx=int(row['pred_idx']))
        show_lrp_result(result, axes[row_idx, 0], axes[row_idx, 1])

    plt.tight_layout()
    plt.show()
else:
    print('No validation mistakes found for the selected split.')

## Summary

This notebook now matches the requested end-to-end setup:

1. `ArcFaceModel` is trained directly on images with `freeze_backbone=False`.
2. EfficientNetB3 backbone weights are updated jointly with the projection head.
3. LRP is applied to a full-model image-to-class-score wrapper.
4. Relevance therefore flows through the entire backbone instead of stopping at cached embeddings.